# Multi-Task Continual Learning for Pest Detection using PROB on Kaggle

This notebook implements the training pipeline for **PROB** (Probabilistic Objectness for Open World Object Detection) on the **IP102** pest dataset. 

### Pipeline Overview:
0. **Clone Repository**: Clone your custom GitHub repository.
1. **Environment Setup & Symbolic Links**: Link dataset folders to the writable workspace.
2. **Parse COCO splits**: Create training/validation split `.txt` files from COCO JSON annotations.
3. **Dataset Registration & PyTorch 2.6+ Compatibility**: Dynamically register "IP102" dataset and patch `torch.load` calls.
4. **Pretrained Checkpoint Optimization**: Reset the starting epoch of the MOWODB checkpoint to `0` to enable 1-epoch finetuning.
5. **Build CUDA Operators**: Patch C++/CUDA sources and compile Deformable DETR CUDA operators.
6. **Training & Finetuning**: Run training and exemplar replay selection sequentially for all 4 tasks.

## Step 0: Clone Custom Repository

In [ ]:
# Clone your custom PROB repository and enter its directory
!git clone https://github.com/nta2112/PROB-IP102-custom.git
%cd PROB-IP102-custom

## Step 1: Setup Directories and Symbolic Links

In [ ]:
# Create the directories required for the dataset
!mkdir -p data/OWOD/ImageSets/IP102
!mkdir -p models
!mkdir -p exps/IP102/PROB

# Create symbolic links to save disk space and time
!ln -sf /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/JPEGImages data/OWOD/JPEGImages
!ln -sf /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/Annotations data/OWOD/Annotations

print("Directory structure and symlinks successfully created!")

## Step 2: Parse COCO JSON to Pascal VOC Splits (.txt files)

In [ ]:
import json
import os
from pathlib import Path

def extract_stems_from_json(json_path, output_txt_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    stems = []
    for img in data['images']:
        file_name = img['file_name']
        # Extract stem (IP000000000.jpg -> IP000000000)
        stem = Path(file_name).stem
        stems.append(stem)
        
    with open(output_txt_path, 'w') as f:
        for stem in stems:
            f.write(f"{stem}\n")
            
    print(f"Extracted {len(stems)} images from {json_path} to {output_txt_path}")

# Process splits
extract_stems_from_json('/kaggle/input/datasets/nta212/ip102-for-object-detection/train.json', 'data/OWOD/ImageSets/IP102/train.txt')
extract_stems_from_json('/kaggle/input/datasets/nta212/ip102-for-object-detection/val.json', 'data/OWOD/ImageSets/IP102/val.txt')
extract_stems_from_json('/kaggle/input/datasets/nta212/ip102-for-object-detection/test.json', 'data/OWOD/ImageSets/IP102/test.txt')

## Step 3: Register IP102 Dataset and Task Splits + Patch torch.load

In [ ]:
# Read and parse class names
classes_file = '/kaggle/input/datasets/nta212/ip102-for-object-detection/classes.txt'
with open(classes_file, 'r') as f:
    class_lines = [line.strip().split(maxsplit=1) for line in f.readlines() if line.strip()]

# Sort classes by their integer ID mapping
class_lines.sort(key=lambda x: int(x[0]))
classes = [c[1] for c in class_lines]

# Split classes into 4 tasks (7, 6, 6, 6)
t1 = classes[:7]
t2 = classes[7:13]
t3 = classes[13:19]
t4 = classes[19:]

print("T1 Classes:", t1)
print("T2 Classes:", t2)
print("T3 Classes:", t3)
print("T4 Classes:", t4)

# 1. Programmatically register dataset to open_world.py
open_world_py = 'datasets/torchvision_datasets/open_world.py'
with open(open_world_py, 'r') as f:
    content = f.read()

ip102_class_definition = f"""
# Auto-generated for IP102 pest detection
IP102_T1_CLASS_NAMES = {t1}
IP102_T2_CLASS_NAMES = {t2}
IP102_T3_CLASS_NAMES = {t3}
IP102_T4_CLASS_NAMES = {t4}
VOC_COCO_CLASS_NAMES["IP102"] = tuple(itertools.chain(IP102_T1_CLASS_NAMES, IP102_T2_CLASS_NAMES, IP102_T3_CLASS_NAMES, IP102_T4_CLASS_NAMES, UNK_CLASS))
"""

if 'VOC_COCO_CLASS_NAMES["IP102"]' not in content:
    content = content.replace(
        'VOC_COCO_CLASS_NAMES["TOWOD"] = tuple(itertools.chain(VOC_CLASS_NAMES, T2_CLASS_NAMES, T3_CLASS_NAMES, T4_CLASS_NAMES, UNK_CLASS))',
        'VOC_COCO_CLASS_NAMES["TOWOD"] = tuple(itertools.chain(VOC_CLASS_NAMES, T2_CLASS_NAMES, T3_CLASS_NAMES, T4_CLASS_NAMES, UNK_CLASS))\\n' + ip102_class_definition
    )
    with open(open_world_py, 'w') as f:
        f.write(content)
    print("Registered 'IP102' dataset in open_world.py!")
else:
    print("'IP102' dataset already registered!")

# 2. Patch main_open_world.py for PyTorch 2.6+ compatibility (weights_only=False)
main_open_world_py = 'main_open_world.py'
with open(main_open_world_py, 'r') as f:
    main_content = f.read()

patch_code = """
# Patch torch.load for PyTorch 2.6+ compatibility
import functools
torch.load = functools.partial(torch.load, weights_only=False)
"""

if 'weights_only=False' not in main_content and 'Patch torch.load' not in main_content:
    main_content = main_content.replace('import torch', 'import torch\\n' + patch_code)
    with open(main_open_world_py, 'w') as f:
        f.write(main_content)
    print("Patched main_open_world.py to set weights_only=False for torch.load compatibility!")
else:
    print("main_open_world.py already patched!")

## Step 4: Reset Epoch Counter in Pretrained Checkpoint

In [ ]:
import torch

pretrained_path = '/kaggle/input/models/chienkhu/pretrain-prob/pytorch/default/1/MOWODB/t4.pth'
output_reset_path = '/kaggle/working/t4_reset.pth'

# Specifying weights_only=False to allow loading custom classes like argparse.Namespace in PyTorch 2.6+
checkpoint = torch.load(pretrained_path, map_location='cpu', weights_only=False)
checkpoint['epoch'] = -1  # Resetting epoch so start_epoch becomes 0
torch.save(checkpoint, output_reset_path)
print(f"Pretrained checkpoint reset saved to {output_reset_path}!")

## Step 5: Download DINO Backbone, Patch C++/CUDA Sources & Build Operators

In [ ]:
# Download the self-supervised backbone
!wget -q -P models/ https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth

# Programmatically patch the CUDA C++ source files to be compatible with PyTorch 2.x
def patch_cuda_sources():
    cuda_file = 'models/ops/src/cuda/ms_deform_attn_cuda.cu'
    with open(cuda_file, 'r') as f:
        code = f.read()
    code = code.replace('.type().is_cuda()', '.is_cuda()')
    code = code.replace('value.type()', 'value.scalar_type()')
    with open(cuda_file, 'w') as f:
        f.write(code)
    
    h_file = 'models/ops/src/ms_deform_attn.h'
    with open(h_file, 'r') as f:
        code = f.read()
    code = code.replace('value.type().is_cuda()', 'value.is_cuda()')
    with open(h_file, 'w') as f:
        f.write(code)
        
    print("Successfully patched C++ and CUDA source files for PyTorch 2.x compatibility!")

patch_cuda_sources()

# Compile the CUDA operators
%cd models/ops
!python setup.py build install
%cd ../..

## Step 6: Execute Training Pipeline (1 Epoch per Stage)

In [ ]:
# Configure WandB project and offline mode if preferred
import os
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_API_KEY"] = "0000000000000000000000000000000000000000"

### Task 1: Train on classes 0–6 (7 classes)

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t1" --dataset IP102 --PREV_INTRODUCED_CLS 0 --CUR_INTRODUCED_CLS 7 \
    --train_set 'train' --test_set 'test' --epochs 1 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --obj_temp 1.3 \
    --wandb_name "PROB_IP102_t1" --exemplar_replay_selection --exemplar_replay_max_length 850 \
    --exemplar_replay_dir "PROB_IP102" --exemplar_replay_cur_file "learned_owod_t1_ft.txt" \
    --pretrain "/kaggle/working/t4_reset.pth"

### Task 2: Train on classes 7–12 (6 classes)

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t2" --dataset IP102 --PREV_INTRODUCED_CLS 7 --CUR_INTRODUCED_CLS 6 \
    --train_set 'train' --test_set 'test' --epochs 2 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --obj_temp 1.3 --freeze_prob_model \
    --wandb_name "PROB_IP102_t2" \
    --exemplar_replay_selection --exemplar_replay_max_length 1743 --exemplar_replay_dir "PROB_IP102" \
    --exemplar_replay_prev_file "learned_owod_t1_ft.txt" --exemplar_replay_cur_file "learned_owod_t2_ft.txt" \
    --pretrain "exps/IP102/PROB/t1/checkpoint0000.pth" --lr 2e-5

### Task 2 Finetune

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t2_ft" --dataset IP102 --PREV_INTRODUCED_CLS 7 --CUR_INTRODUCED_CLS 6 \
    --train_set "PROB_IP102/learned_owod_t2_ft" --test_set 'test' --epochs 3 --lr_drop 40 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --obj_temp 1.3 \
    --wandb_name "PROB_IP102_t2_ft" \
    --pretrain "exps/IP102/PROB/t2/checkpoint0001.pth"

### Task 3: Train on classes 13–18 (6 classes)

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t3" --dataset IP102 --PREV_INTRODUCED_CLS 13 --CUR_INTRODUCED_CLS 6 \
    --train_set 'train' --test_set 'test' --epochs 4 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --freeze_prob_model --obj_temp 1.3 \
    --wandb_name "PROB_IP102_t3" \
    --exemplar_replay_selection --exemplar_replay_max_length 2361 --exemplar_replay_dir "PROB_IP102" \
    --exemplar_replay_prev_file "learned_owod_t2_ft.txt" --exemplar_replay_cur_file "learned_owod_t3_ft.txt" \
    --pretrain "exps/IP102/PROB/t2_ft/checkpoint0002.pth" --lr 2e-5

### Task 3 Finetune

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t3_ft" --dataset IP102 --PREV_INTRODUCED_CLS 13 --CUR_INTRODUCED_CLS 6 \
    --train_set "PROB_IP102/learned_owod_t3_ft" --test_set 'test' --epochs 5 --lr_drop 35 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --obj_temp 1.3 \
    --wandb_name "PROB_IP102_t3_ft" \
    --pretrain "exps/IP102/PROB/t3/checkpoint0003.pth"

### Task 4: Train on classes 19–24 (6 classes)

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t4" --dataset IP102 --PREV_INTRODUCED_CLS 19 --CUR_INTRODUCED_CLS 6 \
    --train_set 'train' --test_set 'test' --epochs 6 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --freeze_prob_model --obj_temp 1.3 \
    --wandb_name "PROB_IP102_t4" \
    --exemplar_replay_selection --exemplar_replay_max_length 2749 --exemplar_replay_dir "PROB_IP102" \
    --exemplar_replay_prev_file "learned_owod_t3_ft.txt" --exemplar_replay_cur_file "learned_owod_t4_ft.txt" \
    --num_inst_per_class 40 \
    --pretrain "exps/IP102/PROB/t3_ft/checkpoint0004.pth" --lr 2e-5

### Task 4 Finetune

In [ ]:
!python -u main_open_world.py \
    --output_dir "exps/IP102/PROB/t4_ft" --dataset IP102 --PREV_INTRODUCED_CLS 19 --CUR_INTRODUCED_CLS 6 \
    --train_set "PROB_IP102/learned_owod_t4_ft" --test_set 'test' --epochs 7 --lr_drop 50 --num_classes 26 \
    --model_type 'prob' --obj_loss_coef 8e-4 --obj_temp 1.3 \
    --wandb_name "PROB_IP102_t4_ft" \
    --pretrain "exps/IP102/PROB/t4/checkpoint0005.pth"